In this competition, you’ll gain access to two similar datasets that include passenger information like name, age, gender, socio-economic class, etc. One dataset is titled "train.csv" and the other is titled "test.csv".

"train.csv" will contain the details of a subset of the passengers on board (891 to be exact) and importantly, will reveal whether they survived or not, also known as the “ground truth”.

The "test.csv" dataset contains similar information but does not disclose the “ground truth” for each passenger. It’s your job to predict these outcomes.

Using the patterns you find in the "train.csv" data, predict whether the other 418 passengers on board (found in test.csv) survived.

In [1]:
import pandas as pd
import numpy as np

# dataset with the ground truth
data_train = pd.read_csv("/Users/alvaromartinezmartinez/Desktop/Python/MLJupyter/train_titanic.csv")

# dataset for which we need to predict the survival
data_test = pd.read_csv("/Users/alvaromartinezmartinez/Desktop/Python/MLJupyter/test_titanic.csv")

El objetivo es determinar "Survived" para cada "PassengerId" del fichero "train.csv".

## Gestión de los datos inicial

In [2]:
data_train.columns

Index(['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp',
       'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked'],
      dtype='object')

In [3]:
data_test.columns

Index(['PassengerId', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch',
       'Ticket', 'Fare', 'Cabin', 'Embarked'],
      dtype='object')

In [4]:
data_train.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [5]:
data_test.head()

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,892,3,"Kelly, Mr. James",male,34.5,0,0,330911,7.8292,NaN,Q
1,893,3,"Wilkes, Mrs. James (Ellen Needs)",female,47.0,1,0,363272,7.0000,NaN,S
2,894,2,"Myles, Mr. Thomas Francis",male,62.0,0,0,240276,9.6875,NaN,Q
3,895,3,"Wirz, Mr. Albert",male,27.0,0,0,315154,8.6625,NaN,S
4,896,3,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",female,22.0,1,1,3101298,12.2875,NaN,S


Definición de variables:
- passengerid
- survival: 0 = No, 1 = Yes
- pclass (Ticket class): 1 = 1st, 2 = 2nd, 3 = 3rd
- name
- sex
- age: in years
- sibsp: # of siblings / spouses aboard the Titanic
- parch: # of parents / children aboard the Titanic
- ticket: ticket number
- fare
- cabin: cabin number
- embarked (port of embarkation): C = Cherbourg, Q = Queenstown, S = Southampton

Ahora quiero ver qué columnas tienen huecos vacíos para poder quitarlas o imputarlas.

In [6]:
# Shape of training data (num_rows, num_columns)
print(data_train.shape)

# Number of missing values in each column of training data
missing_val_count_by_column = (data_train.isnull().sum())
print(missing_val_count_by_column[missing_val_count_by_column > 0])

(891, 12)
Age         177
Cabin       687
Embarked      2
dtype: int64


Vemos que la columna de cabina está practicamente vacía, así que esta la quitaremos entera.

La de embarked está practicamente llena, solo que no tiene practicamente relevancia el lugar de embarque, así que la quito.

Quito también las columnas de Name y Ticket ya que no tienen importancia.

La de edad la podemos imputar.

In [7]:
from sklearn.impute import SimpleImputer

# Drop cabin, embarked, name and ticket columns
reduced_data_train = data_train.drop(['Cabin','Embarked','Name','Ticket'], axis=1)
reduced_data_test = data_test.drop(['Cabin','Embarked','Name','Ticket'],axis=1)

# Imputation
my_imputer = SimpleImputer()
reduced_data_train["Age"] = my_imputer.fit_transform(reduced_data_train[["Age"]])


Ahora quiero hacer ordinal encoding tranformando Female = 0 y Male = 1.

In [8]:
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()
reduced_data_train["Sex"] = encoder.fit_transform(reduced_data_train["Sex"])
reduced_data_test["Sex"] = encoder.fit_transform(reduced_data_test["Sex"])

In [9]:
reduced_data_train.head()

,PassengerId,Survived,Pclass,Sex,Age,SibSp,Parch,Fare
0,1,0,3,1,22.0,1,0,7.2500
1,2,1,1,0,38.0,1,0,71.2833
2,3,1,3,0,26.0,0,0,7.9250
3,4,1,1,0,35.0,1,0,53.1000
4,5,0,3,1,35.0,0,0,8.0500


In [10]:
reduced_data_test.head()

,PassengerId,Pclass,Sex,Age,SibSp,Parch,Fare
0,892,3,1,34.5,0,0,7.8292
1,893,3,0,47.0,1,0,7.0000
2,894,2,1,62.0,0,0,9.6875
3,895,3,1,27.0,0,0,8.6625
4,896,3,0,22.0,1,1,12.2875


Comprobamos que efectivamente ya no tenemos huecos vacíos en la tabla de train:

In [11]:
# Number of missing values in each column of training data
missing_val_count_by_column = (reduced_data_train.isnull().sum())
print(missing_val_count_by_column[missing_val_count_by_column > 0])

Series([], dtype: int64)


## Random Forest

Utilizamos un random forest como modelo dada su sencillez para crearlo.

In [12]:
from sklearn.tree import DecisionTreeClassifier # Classifier para discreto, Regressor para continuo

# Guardamos el predicition target 
y_train = reduced_data_train.Survived 
X_train = reduced_data_train[['Pclass','Sex','Age','SibSp','Parch','Fare']] # Sin passengerid y sin survived

# Define model. Specify a number for random_state to ensure same results each run
titanic_model = DecisionTreeClassifier(random_state=1)

# Fit model
titanic_model.fit(X_train, y_train)

DecisionTreeClassifier(random_state=1)

In [13]:
X_test = reduced_data_test[['Pclass','Sex','Age','SibSp','Parch','Fare']]

predictions = titanic_model.predict(X_test)

print("Making predictions for the following people:")
print(X_test.head())
print("The predictions are")
print(titanic_model.predict(X_test.head()))

Making predictions for the following people:
   Pclass  Sex   Age  SibSp  Parch     Fare
0       3    1  34.5      0      0   7.8292
1       3    0  47.0      1      0   7.0000
2       2    1  62.0      0      0   9.6875
3       3    1  27.0      0      0   8.6625
4       3    0  22.0      1      1  12.2875
The predictions are
[0 0 1 1 1]


## Saving the results for the Random Forest

In [14]:
solution = pd.DataFrame({'PassengerId': reduced_data_test.PassengerId, 'Survived': predictions})
solution.to_csv('submission.csv', index=False)

colsurv_rf = solution['Survived']

## XGBoost

In [15]:
from xgboost import XGBClassifier

titanic_xgmodel = XGBClassifier(n_estimators=500, use_label_encoder=False, eval_metric="logloss")
titanic_xgmodel.fit(X_train, y_train)

/opt/anaconda3/lib/python3.13/site-packages/xgboost/training.py:183: UserWarning: [12:38:54] WARNING: /var/folders/sy/f16zz6x50xz3113nwtb9bvq00000gp/T/abs_7eg2b3w9sf/croot/xgboost-split_1749630922962/work/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=None, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=None, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=500, n_jobs=None,
              num_parallel_tree=None, ...)

In [16]:
predictions_xgb = titanic_xgmodel.predict(X_test)

print("Making predictions for the following people:")
print(X_test.head())
print("The predictions are")
print(titanic_xgmodel.predict(X_test.head()))

Making predictions for the following people:
   Pclass  Sex   Age  SibSp  Parch     Fare
0       3    1  34.5      0      0   7.8292
1       3    0  47.0      1      0   7.0000
2       2    1  62.0      0      0   9.6875
3       3    1  27.0      0      0   8.6625
4       3    0  22.0      1      1  12.2875
The predictions are
[0 0 1 1 1]


## Saving the solutions for XGBoost

In [17]:
solution = pd.DataFrame({'PassengerId': reduced_data_test.PassengerId, 'Survived': predictions_xgb})
solution.to_csv('submission.csv', index=False)

colsurv_xgb = solution['Survived']

## Improved XGBoost

Para validar mejor el modelo me hace falta un X_valid y un y_valid que voy a sacar de X_train y y_train.

In [18]:
from sklearn.model_selection import train_test_split

# Dividimos el train original en train + valid
X_train_new, X_valid, y_train_new, y_valid = train_test_split(X_train, y_train, test_size=0.2, random_state=42)

In [19]:
titanic_xgmodel_new = XGBClassifier(
    n_estimators=500,        # número de árboles
    learning_rate=0.05,      # tamaño de los pasos de boosting
    max_depth=4,             # profundidad máxima de cada árbol
    subsample=0.8,           # fracción de datos por árbol
    colsample_bytree=0.8,    # fracción de features por árbol
    use_label_encoder=False,
    eval_metric="logloss",
    early_stopping_rounds=20, 
    verbose=False)

titanic_xgmodel_new.fit(X_train_new, y_train_new, eval_set=[(X_valid, y_valid)])


[0]	validation_0-logloss:0.67171
[1]	validation_0-logloss:0.66335
[2]	validation_0-logloss:0.65618
[3]	validation_0-logloss:0.64970
[4]	validation_0-logloss:0.63084
[5]	validation_0-logloss:0.62508
[6]	validation_0-logloss:0.60850
[7]	validation_0-logloss:0.59314
[8]	validation_0-logloss:0.57899
[9]	validation_0-logloss:0.56722
[10]	validation_0-logloss:0.56319
[11]	validation_0-logloss:0.55954
[12]	validation_0-logloss:0.55641
[13]	validation_0-logloss:0.54502
[14]	validation_0-logloss:0.54170
[15]	validation_0-logloss:0.53294
[16]	validation_0-logloss:0.52895
[17]	validation_0-logloss:0.52097
[18]	validation_0-logloss:0.51279
[19]	validation_0-logloss:0.50974
[20]	validation_0-logloss:0.50336
[21]	validation_0-logloss:0.50044
[22]	validation_0-logloss:0.49348
[23]	validation_0-logloss:0.48782
[24]	validation_0-logloss:0.48163
[25]	validation_0-logloss:0.47640


/opt/anaconda3/lib/python3.13/site-packages/xgboost/callback.py:386: UserWarning: [12:38:57] WARNING: /var/folders/sy/f16zz6x50xz3113nwtb9bvq00000gp/T/abs_7eg2b3w9sf/croot/xgboost-split_1749630922962/work/src/learner.cc:738: 
Parameters: { "use_label_encoder", "verbose" } are not used.

  self.starting_round = model.num_boosted_rounds()


[26]	validation_0-logloss:0.47486
[27]	validation_0-logloss:0.47403
[28]	validation_0-logloss:0.46930
[29]	validation_0-logloss:0.46509
[30]	validation_0-logloss:0.46154
[31]	validation_0-logloss:0.45825
[32]	validation_0-logloss:0.45611
[33]	validation_0-logloss:0.45419
[34]	validation_0-logloss:0.45154
[35]	validation_0-logloss:0.44921
[36]	validation_0-logloss:0.44695
[37]	validation_0-logloss:0.44436
[38]	validation_0-logloss:0.44348
[39]	validation_0-logloss:0.44091
[40]	validation_0-logloss:0.43894
[41]	validation_0-logloss:0.43732
[42]	validation_0-logloss:0.43584
[43]	validation_0-logloss:0.43493
[44]	validation_0-logloss:0.43381
[45]	validation_0-logloss:0.43332
[46]	validation_0-logloss:0.43176
[47]	validation_0-logloss:0.43061
[48]	validation_0-logloss:0.43022
[49]	validation_0-logloss:0.42975
[50]	validation_0-logloss:0.42973
[51]	validation_0-logloss:0.42974
[52]	validation_0-logloss:0.42869
[53]	validation_0-logloss:0.42768
[54]	validation_0-logloss:0.42748
[55]	validatio

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=20,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.05, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=4, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=500, n_jobs=None,
              num_parallel_tree=None, ...)

In [20]:
predictions_xgb_new = titanic_xgmodel_new.predict(X_test)

print("Making predictions for the following people:")
print(X_test.head())
print("The predictions are")
print(titanic_xgmodel_new.predict(X_test.head()))

Making predictions for the following people:
   Pclass  Sex   Age  SibSp  Parch     Fare
0       3    1  34.5      0      0   7.8292
1       3    0  47.0      1      0   7.0000
2       2    1  62.0      0      0   9.6875
3       3    1  27.0      0      0   8.6625
4       3    0  22.0      1      1  12.2875
The predictions are
[0 0 0 0 1]


## Saving solutions for the improved XGBoost

In [21]:
solution = pd.DataFrame({'PassengerId': reduced_data_test.PassengerId, 'Survived': predictions_xgb_new})
solution.to_csv('submission.csv', index=False)

colsurv_imprxgb = solution['Survived']

El resultado ha sido $78\%$ de aciertos.

## Comparacion de resultados

In [22]:
perfect_solution = pd.read_csv("/Users/alvaromartinezmartinez/Desktop/Python/MLJupyter/checksubmission_titanic.csv")

colsurv_perf = perfect_solution['Survived']

In [23]:
diff_rf = colsurv_rf != colsurv_perf
diff_xgb = colsurv_xgb != colsurv_perf  # True where values differ
diff_imprxgb = colsurv_imprxgb != colsurv_perf

total_cells_rf = diff_rf.size              # total number of cells (418)
total_cells_xgb = diff_xgb.size
total_cells_imprxgb = diff_imprxgb.size

num_differences_rf = diff_rf.sum().sum()
num_differences_xgb = diff_xgb.sum().sum()
num_differences_imprxgb = diff_imprxgb.sum().sum()

percent_equal_rf = 100-((num_differences_rf / total_cells_rf) * 100)
percent_equal_xgb = 100-((num_differences_xgb / total_cells_xgb) * 100)
percent_equal_imprxgb = 100-((num_differences_imprxgb / total_cells_imprxgb) * 100)

print(f"Percentage of equal cells for Random Forest: {percent_equal_rf:.2f}%")
print(f"Percentage of equal cells for XGBoost: {percent_equal_xgb:.2f}%")
print(f"Percentage of equal cells for improved XGBoost: {percent_equal_imprxgb:.2f}%")

Percentage of equal cells for Random Forest: 73.21%
Percentage of equal cells for XGBoost: 73.68%
Percentage of equal cells for improved XGBoost: 77.99%
